[Vortex](https://vortex.dev/) is an extensible ecosystem for compressed columnar data. It provides an efficient file format built around the latest research from the database community. Vortex integrates seamlessly with Apache Arrow, enabling zero-copy conversion between Vortex and Arrow. This makes it easy to combine Vortex with [ADBC](https://arrow.apache.org/adbc/) (Arrow Database Connectivity), a cross-language, Arrow-native API for database access. ADBC lets you execute SQL queries and retrieve results directly in Arrow format, which can then be written to Vortex files or read back for ingestion into databases.

In this notebook, you will:
1. Set up an ADBC connection using DuckDB as the database driver.
2. Execute a SQL query and write the Arrow results to a Vortex file.
3. Read the Vortex file back as Arrow data and ingest it into a database table via ADBC.

Note: This notebook uses [DuckDB](https://duckdb.org/) for simplicity, but ADBC supports a wide range of SQL databases including Snowflake, Databricks, BigQuery, Dremio, Trino, PostgreSQL, MySQL, SQLite, and more.

## Setup

Install the required dependencies:

In [1]:
%pip install -q dbc adbc-driver-manager pyarrow vortex-data

Note: you may need to restart the kernel to use updated packages.


Install the DuckDB ADBC driver:

In [ ]:
!dbc install -q duckdb

Import Vortex and the ADBC driver manager:

In [3]:
import vortex as vx
from adbc_driver_manager import dbapi

Create a DBAPI-style connection with DuckDB (in-memory) and cursor:

In [4]:
connection = dbapi.connect(driver="duckdb", autocommit=True)
cursor = connection.cursor()

## Write Query Results to Vortex

Execute a SQL query and get the results as a PyArrow Table:

In [5]:
cursor.execute(
    "SELECT * FROM read_csv('https://blobs.duckdb.org/data/penguins.csv', nullstr = 'NA')"
)
table = cursor.fetch_arrow_table()

Write the results to a Vortex file:

In [6]:
vx.io.write(table, "penguins.vortex")

## Ingest Vortex into Database

Read a Vortex file as an Arrow record batch reader:

In [7]:
reader = vx.open("penguins.vortex").to_arrow()

Ingest the reader into a database table:

In [8]:
cursor.adbc_ingest(table_name="penguins", data=reader)

344

Query the ingested data:

In [9]:
cursor.execute("SELECT * FROM penguins")
cursor.fetch_arrow_table()

pyarrow.Table
species: string
island: string
bill_length_mm: double
bill_depth_mm: double
flipper_length_mm: int64
body_mass_g: int64
sex: string
year: int64
----
species: [["Adelie","Adelie","Adelie","Adelie","Adelie",...,"Chinstrap","Chinstrap","Chinstrap","Chinstrap","Chinstrap"]]
island: [["Torgersen","Torgersen","Torgersen","Torgersen","Torgersen",...,"Dream","Dream","Dream","Dream","Dream"]]
bill_length_mm: [[39.1,39.5,40.3,null,36.7,...,55.8,43.5,49.6,50.8,50.2]]
bill_depth_mm: [[18.7,17.4,18,null,19.3,...,19.8,18.1,18.2,19,18.7]]
flipper_length_mm: [[181,186,195,null,193,...,207,202,193,210,198]]
body_mass_g: [[3750,3800,3250,null,3450,...,4000,3400,3775,4100,3775]]
sex: [["male","female","female",null,"female",...,"male","female","male","male","female"]]
year: [[2007,2007,2007,2007,2007,...,2009,2009,2009,2009,2009]]

## Cleanup

Close the cursor and connection:

In [10]:
cursor.close()
connection.close()

Optionally, delete the Vortex file:

In [11]:
from pathlib import Path

Path("penguins.vortex").unlink(missing_ok=True)